<div dir="rtl" align="right">

# قدرةُ نطاقاتِ موجاتِ الدماغِ \(Band Power\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَستخدمُ طريقةَ ويلش (Welch) لتقديرِ الكثافةِ الطيفيّةِ لقدرةِ الإشارة، ثمّ نُكاملُ القدرةَ في كلِّ نطاقٍ تردديٍّ (دلتا، ثيتا، ألفا، بيتا، غاما) ونُقارنُها كنسبٍ مئويّةٍ.

## المُخرجاتُ المُتوقّعةُ

- منحنى PSD يُظهرُ توزيعَ القدرةِ عبرَ التردداتِ
- مخططٌ شريطيٌّ يُقارنُ القدرةَ النسبيّةَ في كلِّ نطاقٍ
- تَركزٌ في النطاقاتِ المنخفضةِ (دلتا، ثيتا) بسببِ قانون 1/f

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| معدّلُ الأخذِ | 200 Hz | عيّنةٌ كلَّ 5 ms |
| nperseg | 1024 | حجمُ نافذةِ ويلش |
| النطاقاتُ | 5 | دلتا، ثيتا، ألفا، بيتا، غاما |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb pywt


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. حسابُ قدرةِ النطاقاتِ

نَستخدمُ `scipy.signal.welch` لتقديرِ الكثافةِ الطيفيّةِ لقدرةِ الإشارة. ثمّ نُكاملُ القدرةَ في كلِّ نطاقٍ تردديٍّ باستخدامِ `np.trapezoid`.

</div>

In [ ]:
from scipy.signal import welch

freqs, psd = welch(channel_data, fs=fs, nperseg=1024)

BANDS = [
    ('Delta', 0.5, 4),
    ('Theta', 4, 8),
    ('Alpha', 8, 13),
    ('Beta', 13, 30),
    ('Gamma', 30, 80),
]

band_powers = {}
for name, fmin, fmax in BANDS:
    band_mask = (freqs >= fmin) & (freqs <= fmax)
    power = np.trapezoid(psd[band_mask], freqs[band_mask])
    band_powers[name] = power

total_power = sum(band_powers.values())
relative_powers = {k: v / total_power * 100 for k, v in band_powers.items()}
for name, power in relative_powers.items():
    print(f'{name}: {power:.1f}%')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- منحنى PSD يُظهرُ توزيعَ القدرةِ على مقياسٍ لوغاريتميٍّ
- التظليلُ المُلوّنُ يُحدّدُ نطاقاتِ موجاتِ الدماغِ الخمسَ
- المخططُ الشريطيُّ يُقارنُ القدرةَ النسبيّةَ كنسبٍ مئويّةٍ


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BAND_COLORS = ['green', 'blue', 'orange', 'red', 'purple']

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('Power Spectral Density - Channel P4',
                                    'Relative Band Power - Channel P4'))
fig.add_trace(go.Scatter(x=freqs, y=psd, name='PSD',
                         line=dict(color='black', width=1)), row=1, col=1)
for (name, fmin, fmax), color in zip(BANDS, BAND_COLORS):
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=1, col=1)
fig.update_xaxes(range=[0, 80], row=1, col=1)
fig.update_yaxes(type='log', row=1, col=1)

names = list(relative_powers.keys())
values = list(relative_powers.values())
fig.add_trace(go.Bar(x=names, y=values, marker_color=BAND_COLORS,
                     name='Relative Power'), row=2, col=1)

fig.update_layout(height=800, title_text='Brain Wave Band Power - Channel P4',
                  xaxis_title='Frequency (Hz)', xaxis2_title='Band',
                  yaxis_title='PSD (uV^2/Hz)', yaxis2_title='Relative Power (%)',
                  showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- طريقةُ ويلش تُقدّرُ الكثافةَ الطيفيّةَ لقدرةِ الإشارةِ بكفاءةٍ
- القدرةُ النسبيّةُ تُسهّلُ المقارنةَ بينَ النطاقاتِ وتُلغي أثرَ السعةِ المُطلقةِ
- في بياناتِ EEG الخام، تَتركّزُ القدرةُ في النطاقاتِ المنخفضةِ (قانون 1/f)
- يُنصحُ بتطبيقِ مرشّحِ تمريرِ النطاقِ قبلَ حسابِ القدرةِ لِإزالةِ الآثارِ الشائبةِ


</div>